In [2]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
import subprocess
import sys

# 1. Install dependencies if missing BEFORE importing them
try:
    import ultralytics
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "timm"], check=True)
    import ultralytics
    import timm

from ultralytics import YOLO
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# PATH A: HARDENED INFERENCE CONFIGURATION
# ==========================================
# Use the exact same dataset path you used for the baseline 0.5401 score
dataset_path = "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb"

# We will sweep across multiple thresholds to find the optimum!
CONFIDENCE_THRESHOLDS = [0.25, 0.40, 0.50, 0.65, 0.80]
# ==========================================

print(f"Scanning dataset at: {dataset_path}")

# Check for weights in typical kaggle paths
weight_dir_options = [
    "/kaggle/input/chakramodel-weights",
    "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
    "/kaggle/working",
    "."
]

yolo_weight = None
vit_weight = None

for d in weight_dir_options:
    if os.path.exists(os.path.join(d, "best.pt")):
        yolo_weight = os.path.join(d, "best.pt")
    if os.path.exists(os.path.join(d, "chakra_transformer_best.pth")):
        vit_weight = os.path.join(d, "chakra_transformer_best.pth")

if not yolo_weight or not vit_weight:
    print("ERROR: Weights not found in Kaggle! Please upload best.pt and chakra_transformer_best.pth")
    sys.exit(1)

print(f"Found YOLO weight at {yolo_weight}")
print(f"Found ViT weight at {vit_weight}")

# 3. Model definitions
class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, backbone_name='vit_large_patch16_384', pretrained=False, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=False)
        self.embed_dim = self.backbone.embed_dim
        self.decode_head = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 256, kernel_size=4, stride=4),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1),
        )
        self.dropout1 = nn.Dropout2d(p=0.5)
        self.dropout2 = nn.Dropout2d(p=0.5)

    def forward(self, x):
        b, c, h, w = x.shape
        features = self.backbone.forward_features(x)
        if features.dim() == 3:
            grid_h, grid_w = h // 16, w // 16
            expected = grid_h * grid_w
            if features.shape[1] == expected + 1:
                features = features[:, 1:, :]
            elif features.shape[1] > expected:
                features = features[:, :expected, :]
            features = features.transpose(1, 2).contiguous().view(b, self.embed_dim, grid_h, grid_w)

        x_dec = features
        for i, layer in enumerate(self.decode_head):
            x_dec = layer(x_dec)
            if i == 2: x_dec = self.dropout1(x_dec)
            elif i == 5: x_dec = self.dropout2(x_dec)

        logits = x_dec
        if logits.shape[2:] != (h, w):
            logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
        return logits


def letterbox(img, new_shape=(384, 384), color=(0, 0, 0)):
    shape = img.shape[:2]
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    dw, dh = dw / 2, dh / 2
    if shape[::-1] != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return img, (dw, dh, r)


def calculate_dice(pred, target):
    smooth = 1e-5
    pred_f = pred.flatten()
    target_f = target.flatten()
    intersection = np.sum(pred_f * target_f)
    return (2. * intersection + smooth) / (np.sum(pred_f) + np.sum(target_f) + smooth)


# ---------------------------------------------------------
# Fix: generalized mask<->image matching.
# Instead of hardcoding a couple of suffixes, strip any known
# mask-indicator token from EITHER end of the mask filename and
# compare the normalized base against image bases. Covers
# "_mask", "_gt", "_manual", "_label", "_seg", "_segmentation",
# "mask_", "gt_", "label_" and combinations, without risking the
# old numeric-substring collisions (e.g. "1" matching "10").
# ---------------------------------------------------------
MASK_TOKENS = ["mask", "ground", "gt", "label", "seg", "segmentation", "manual"]


def normalize_mask_base(filename_base):
    base = filename_base
    changed = True
    while changed:
        changed = False
        low = base.lower()
        for tok in MASK_TOKENS:
            for pattern in (f"_{tok}", f"-{tok}", tok):
                if low.startswith(pattern):
                    base = base[len(pattern):]
                    low = base.lower()
                    changed = True
                elif low.endswith(pattern):
                    base = base[: -len(pattern)]
                    low = base.lower()
                    changed = True
        base = base.strip("_- ")
    return base


print("Loading YOLO stage 1...")
yolo_model = YOLO(yolo_weight)

print("Loading ChakraTransformer stage 2...")
segmenter = ChakraTransformerSegmenter('vit_large_patch16_384', pretrained=False, num_classes=1)
state_dict = torch.load(vit_weight, map_location='cpu', weights_only=True)
if "model" in state_dict: state_dict = state_dict["model"]
new_state_dict = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in state_dict.items()}

load_result = segmenter.load_state_dict(new_state_dict, strict=False)
print("ViT Weights Load Result:")
if load_result.missing_keys:
    print(f"  Missing Keys: {len(load_result.missing_keys)}")
if load_result.unexpected_keys:
    print(f"  Unexpected Keys: {len(load_result.unexpected_keys)}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
segmenter.to(DEVICE)
segmenter.eval()

# Precompute normalization tensors once
norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

# Generalized dataset loading
all_files = []
for root, _, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
            all_files.append(os.path.join(root, f))

mask_files = []
img_files = []
for f in all_files:
    low = os.path.basename(f).lower()
    if any(tok in low for tok in ("mask", "ground", "gt", "label")):
        mask_files.append(f)
    else:
        img_files.append(f)

# Build lookup: normalized mask base -> mask path
mask_lookup = {}
unresolved_masks = []
for m in mask_files:
    m_base = os.path.splitext(os.path.basename(m))[0]
    norm = normalize_mask_base(m_base)
    if norm in mask_lookup:
        unresolved_masks.append(m)  # collision, skip to avoid mispairing
    else:
        mask_lookup[norm] = m

matched_pairs = []
unmatched_images = []
for img_path in img_files:
    base = os.path.splitext(os.path.basename(img_path))[0]
    matched_mask = mask_lookup.get(base)
    if matched_mask:
        matched_pairs.append((img_path, matched_mask))
    else:
        unmatched_images.append(img_path)

print(f"Successfully matched {len(matched_pairs)} image-mask pairs.")
if unmatched_images:
    print(f"WARNING: {len(unmatched_images)} images had no mask match "
          f"(e.g. {[os.path.basename(p) for p in unmatched_images[:5]]})")
if unresolved_masks:
    print(f"WARNING: {len(unresolved_masks)} masks had ambiguous/colliding normalized names "
          f"and were skipped (e.g. {[os.path.basename(p) for p in unresolved_masks[:5]]})")

# Cache inference once at the minimum threshold, sweep thresholds against cached boxes.
# Safe because Ultralytics NMS only ever suppresses a box using a higher-confidence
# box already in the candidate pool -- adding lower-confidence boxes to that pool
# never changes which higher-confidence boxes survive. So filtering the cached
# min-threshold boxes by a higher threshold reproduces exactly what a direct
# higher-threshold run would keep.
results_by_threshold = {t: {"dices": [], "zero_skips": 0} for t in CONFIDENCE_THRESHOLDS}
MIN_THRESH = min(CONFIDENCE_THRESHOLDS)

print(f"\n--- Running cached pipeline inference (YOLO base conf={MIN_THRESH}) ---")

with torch.no_grad():
    for img_path, mask_path in tqdm(matched_pairs):
        img_bgr = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img_bgr is None or gt_mask is None:
            continue

        orig_h, orig_w = gt_mask.shape[:2]

        if np.max(gt_mask) == 1:
            gt_mask = gt_mask * 255
        gt_bin = (gt_mask > 127).astype(np.uint8)

        # Negative (no-polyp) frames are kept in the eval; calculate_dice's
        # smoothing term correctly scores an all-zero pred against an
        # all-zero gt as dice ~= 1.0.

        img_resized = cv2.resize(img_bgr, (384, 384))
        yolo_results = yolo_model(img_resized, conf=MIN_THRESH, imgsz=384, verbose=False, device=DEVICE)[0]

        cached_masks = []
        for box_data in yolo_results.boxes:
            conf = float(box_data.conf[0].cpu())
            box = box_data.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)

            scale_x = orig_w / 384.0
            scale_y = orig_h / 384.0

            gx, gy, gx2, gy2 = int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)
            gw, gh = gx2 - gx, gy2 - gy

            px, py = int(0.25 * gw), int(0.25 * gh)
            px1, py1 = max(0, gx - px), max(0, gy - py)
            px2, py2 = min(orig_w, gx + gw + px), min(orig_h, gy + gh + py)

            roi_crop = img_bgr[py1:py2, px1:px2]
            if roi_crop.size == 0:
                continue

            roi_crop_resized, (dw, dh, r) = letterbox(roi_crop, new_shape=(384, 384))
            roi_crop_rgb = cv2.cvtColor(roi_crop_resized, cv2.COLOR_BGR2RGB)

            t_roi = torch.from_numpy(roi_crop_rgb).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE) / 255.0
            t_roi = (t_roi - norm_mean) / norm_std

            out = segmenter(t_roi)
            if isinstance(out, tuple):
                out = out[0]
            prob = torch.sigmoid(out).float().squeeze().cpu().numpy()
            seg_roi_mask = (prob > 0.5).astype(np.uint8)

            top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
            left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
            unpad_h = 384 - top - bottom
            unpad_w = 384 - left - right

            seg_roi_mask_unpad = seg_roi_mask[top:top + unpad_h, left:left + unpad_w]
            seg_roi_mask_resized = cv2.resize(seg_roi_mask_unpad, (px2 - px1, py2 - py1), interpolation=cv2.INTER_NEAREST)

            cached_masks.append({
                "conf": conf,
                "mask": seg_roi_mask_resized,
                "coords": (py1, py2, px1, px2)
            })

        for thresh in CONFIDENCE_THRESHOLDS:
            full_pred = np.zeros((orig_h, orig_w), dtype=np.uint8)
            passed = 0

            for c_mask in cached_masks:
                if c_mask["conf"] >= thresh:
                    py1, py2, px1, px2 = c_mask["coords"]
                    full_pred[py1:py2, px1:px2] = np.maximum(
                        full_pred[py1:py2, px1:px2],
                        c_mask["mask"]
                    )
                    passed += 1

            if passed == 0:
                results_by_threshold[thresh]["zero_skips"] += 1

            dice = calculate_dice(full_pred, gt_bin)
            results_by_threshold[thresh]["dices"].append(dice)

print("\n==============================================")
print("--- Confidence Threshold Sweep Results ---")
print("==============================================")
for thresh in CONFIDENCE_THRESHOLDS:
    avg_dice = np.mean(results_by_threshold[thresh]["dices"]) if results_by_threshold[thresh]["dices"] else 0.0
    zero_skips = results_by_threshold[thresh]["zero_skips"]
    print(f"Threshold: {thresh:.2f} | Dice: {avg_dice:.4f} | Zero Detections: {zero_skips}")
print("==============================================")

Scanning dataset at: /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb
ERROR: Weights not found in Kaggle! Please upload best.pt and chakra_transformer_best.pth


SystemExit: 1

In [3]:
%tb

SystemExit: 1

In [4]:
import os
from collections import defaultdict

# ==========================================================
# FULL KAGGLE INPUT DIAGNOSTIC
# Run this FIRST, right after attaching your datasets, before
# running any training/inference cells. It tells you exactly
# what's mounted, where, and how it's structured -- so path
# mismatches ("weights not found", "dataset not found") get
# caught here instead of failing deep inside a pipeline.
# ==========================================================

ROOT = "/kaggle/input"
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
WEIGHT_EXTS = (".pt", ".pth", ".onnx", ".engine")
MASK_TOKENS = ("mask", "ground", "gt", "label", "seg", "segmentation", "manual")
MAX_SAMPLE_NAMES = 6
MAX_TREE_DEPTH = 4  # how deep to print the folder tree


def human_size(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


def is_image(fname):
    return fname.lower().endswith(IMG_EXTS)


def is_weight(fname):
    return fname.lower().endswith(WEIGHT_EXTS)


def looks_like_mask(fname):
    low = fname.lower()
    return any(tok in low for tok in MASK_TOKENS)


if not os.path.exists(ROOT):
    print(f"!! {ROOT} does not exist in this environment. Are you running this on Kaggle?")
else:
    # ---- 1. Top-level: what datasets are actually attached ----
    top_entries = sorted(os.listdir(ROOT))
    print("=" * 70)
    print(f"TOP-LEVEL CONTENTS OF {ROOT}  ({len(top_entries)} entries)")
    print("=" * 70)
    for e in top_entries:
        full = os.path.join(ROOT, e)
        kind = "DIR " if os.path.isdir(full) else "FILE"
        print(f"  [{kind}] {e}")
    print()

    # ---- 2. Full recursive walk: build stats per top-level dataset ----
    print("=" * 70)
    print("RECURSIVE SCAN (grouped by top-level input)")
    print("=" * 70)

    for top in top_entries:
        top_path = os.path.join(ROOT, top)
        if not os.path.isdir(top_path):
            continue

        print(f"\n--- Dataset: {top} ---")

        all_files = []
        dir_count = 0
        for dirpath, dirnames, filenames in os.walk(top_path):
            dir_count += 1
            for f in filenames:
                all_files.append(os.path.join(dirpath, f))

        total_size = sum(os.path.getsize(f) for f in all_files if os.path.exists(f))
        print(f"  Total files: {len(all_files)}  |  Total dirs: {dir_count}  |  Total size: {human_size(total_size)}")

        # extension breakdown
        ext_counts = defaultdict(int)
        for f in all_files:
            ext = os.path.splitext(f)[1].lower() or "(no ext)"
            ext_counts[ext] += 1
        print("  Extensions:", dict(sorted(ext_counts.items(), key=lambda x: -x[1])))

        # weight files -- exact paths, since these are what the pipeline scripts search for
        weight_files = [f for f in all_files if is_weight(os.path.basename(f))]
        if weight_files:
            print(f"  Weight files found ({len(weight_files)}):")
            for wf in weight_files:
                print(f"    {wf}  ({human_size(os.path.getsize(wf))})")
        else:
            print("  Weight files found: NONE")

        # image/mask split
        image_files = [f for f in all_files if is_image(os.path.basename(f))]
        img_only = [f for f in image_files if not looks_like_mask(os.path.basename(f))]
        mask_only = [f for f in image_files if looks_like_mask(os.path.basename(f))]
        if image_files:
            print(f"  Image-type files: {len(image_files)}  (image-like: {len(img_only)}, mask-like: {len(mask_only)})")
            print(f"    Sample image names:  {[os.path.basename(f) for f in img_only[:MAX_SAMPLE_NAMES]]}")
            if mask_only:
                print(f"    Sample mask names:   {[os.path.basename(f) for f in mask_only[:MAX_SAMPLE_NAMES]]}")

        # YOLO-style structure check (images/train, images/val, labels/train, labels/val)
        yolo_dirs = ["images/train", "images/val", "labels/train", "labels/val"]
        yolo_status = {}
        for yd in yolo_dirs:
            candidate = os.path.join(top_path, yd)
            if os.path.isdir(candidate):
                count = len(os.listdir(candidate))
                yolo_status[yd] = count
        if yolo_status:
            print(f"  YOLO-style folder structure detected: {yolo_status}")

        # folder tree (bounded depth, so huge datasets don't flood output)
        print("  Folder tree (depth-limited):")
        for dirpath, dirnames, filenames in os.walk(top_path):
            depth = dirpath[len(top_path):].count(os.sep)
            if depth > MAX_TREE_DEPTH:
                dirnames[:] = []  # stop descending further
                continue
            indent = "    " + "  " * depth
            print(f"{indent}{os.path.basename(dirpath) or dirpath}/  ({len(filenames)} files)")

    print("\n" + "=" * 70)
    print("PATHS YOUR PIPELINE SCRIPTS ACTUALLY CHECK -- verify these match above")
    print("=" * 70)
    candidate_paths = [
        "/kaggle/input/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-yolo-combo-dataset",
        "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb",
        "/kaggle/working",
    ]
    for p in candidate_paths:
        exists = os.path.exists(p)
        marker = "OK  " if exists else "MISSING"
        print(f"  [{marker}] {p}")
        if exists and os.path.isdir(p):
            try:
                contents = os.listdir(p)
                print(f"           -> contains {len(contents)} entries, e.g. {contents[:5]}")
            except Exception as e:
                print(f"           -> could not list: {e}")

    print("\nDone. Fix any [MISSING] paths above (update the variable in the pipeline script,")
    print("or re-attach the dataset under the expected slug) before running training/inference.")

TOP-LEVEL CONTENTS OF /kaggle/input  (1 entries)
  [DIR ] datasets

RECURSIVE SCAN (grouped by top-level input)

--- Dataset: datasets ---
  Total files: 2964  |  Total dirs: 20  |  Total size: 1.6GB
  Extensions: {'.png': 1736, '.tif': 1224, '.csv': 2, '.pth': 1, '.pt': 1}
  Weight files found (2):
    /kaggle/input/datasets/gokulraj324/chakramodel-weights/chakra_transformer_best.pth  (1.2GB)
    /kaggle/input/datasets/gokulraj324/chakramodel-weights/best.pt  (6.0MB)
  Image-type files: 2960  (image-like: 2960, mask-like: 0)
    Sample image names:  ['173.png', '177.png', '162.png', '186.png', '153.png', '167.png']
  Folder tree (depth-limited):
    datasets/  (0 files)
      gokulrocky/  (0 files)
        endoscene-cvc300-polyp-raw-dataset/  (0 files)
          CVC-300/  (0 files)
            images/  (60 files)
            masks/  (60 files)
      nguyenvoquocduong/  (0 files)
        etis-laribpolypdb/  (0 files)
          images/  (196 files)
          masks/  (196 files)
      ba

In [5]:
import os
import csv
import cv2
import numpy as np
from collections import defaultdict

# ==========================================================
# DEEP KAGGLE DATASET DIAGNOSTIC (A-to-Z)
# Run this FIRST, right after attaching your datasets.
# Goes beyond folder listing: opens sample images/masks,
# reports real dimensions/channels/value ranges, classifies
# each dataset as SEGMENTATION vs YOLO-DETECTION vs UNKNOWN,
# checks weight checkpoints, and previews any CSVs.
# ==========================================================

ROOT = "/kaggle/input"
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
WEIGHT_EXTS = (".pt", ".pth", ".onnx", ".engine")
MASK_TOKENS = ("mask", "ground", "gt", "label", "seg", "segmentation", "manual")
SAMPLE_N = 5
MAX_TREE_DEPTH = 6


def human_size(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


def is_image(fname):
    return fname.lower().endswith(IMG_EXTS)


def is_weight(fname):
    return fname.lower().endswith(WEIGHT_EXTS)


def looks_like_mask_name(name):
    low = name.lower()
    return any(tok in low for tok in MASK_TOKENS)


def inspect_image(path):
    """Return (h, w, channels, dtype, min, max) or None on failure."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        h, w = img.shape
        c = 1
    else:
        h, w, c = img.shape
    return (h, w, c, str(img.dtype), int(img.min()), int(img.max()))


def inspect_weight_file(path):
    """Best-effort peek into a torch checkpoint without crashing the script."""
    try:
        import torch
        obj = torch.load(path, map_location="cpu", weights_only=True)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            if "model" in obj and isinstance(obj["model"], dict):
                inner_keys = list(obj["model"].keys())
                return f"dict with top-level keys {keys[:5]}{'...' if len(keys) > 5 else ''}; " \
                       f"'model' sub-dict has {len(inner_keys)} tensor keys, e.g. {inner_keys[:3]}"
            return f"state_dict with {len(keys)} keys, e.g. {keys[:3]}"
        return f"loaded object of type {type(obj)}"
    except Exception as e:
        return f"could not introspect ({type(e).__name__}: {e})"


def preview_csv(path, n=SAMPLE_N):
    try:
        with open(path, newline="", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            rows = []
            for i, row in enumerate(reader):
                if i >= n:
                    break
                rows.append(row)
        return rows
    except Exception as e:
        return [[f"<could not read: {e}>"]]


def classify_directory_set(files_in_tree):
    """
    Given all file paths under one top-level dataset, decide what kind
    of dataset it looks like: YOLO-detection, segmentation (img+mask),
    plain image collection, or unknown/mixed.
    """
    has_yolo_split = any(
        ("images" + os.sep + "train") in f or ("images" + os.sep + "val") in f
        for f in files_in_tree
    )
    has_label_txt = any(f.lower().endswith(".txt") and "label" in f.lower() for f in files_in_tree)
    has_masks = any(looks_like_mask_name(os.path.basename(f)) for f in files_in_tree if is_image(f))
    has_images = any(is_image(f) and not looks_like_mask_name(os.path.basename(f)) for f in files_in_tree)

    if has_yolo_split and has_label_txt:
        return "YOLO-DETECTION (images/train+val with .txt labels)"
    if has_yolo_split and not has_label_txt:
        return "YOLO-STYLE FOLDERS BUT NO LABEL FILES FOUND (incomplete for training)"
    if has_masks and has_images:
        return "SEGMENTATION (image + mask pairs, no bbox labels)"
    if has_images:
        return "PLAIN IMAGE COLLECTION (no masks, no labels)"
    return "UNKNOWN / NON-IMAGE DATA"


if not os.path.exists(ROOT):
    print(f"!! {ROOT} does not exist. Are you running this on Kaggle?")
else:
    top_entries = sorted(os.listdir(ROOT))
    print("=" * 72)
    print(f"TOP-LEVEL CONTENTS OF {ROOT}  ({len(top_entries)} entries)")
    print("=" * 72)
    for e in top_entries:
        full = os.path.join(ROOT, e)
        print(f"  [{'DIR ' if os.path.isdir(full) else 'FILE'}] {e}")

    # Walk everything once; group by the deepest "dataset-looking" folder
    # (Kaggle convention: /kaggle/input/datasets/<user>/<dataset-slug>/...
    #  or /kaggle/input/<dataset-slug>/...)
    dataset_groups = defaultdict(list)

    for dirpath, dirnames, filenames in os.walk(ROOT):
        for f in filenames:
            full_path = os.path.join(dirpath, f)
            rel = os.path.relpath(full_path, ROOT)
            parts = rel.split(os.sep)
            # group key = up to 3 path segments deep (covers both
            # /input/<slug>/... and /input/datasets/<user>/<slug>/...)
            if parts[0] == "datasets" and len(parts) > 2:
                group_key = os.sep.join(parts[:3])
            else:
                group_key = parts[0]
            dataset_groups[group_key].append(full_path)

    print("\n" + "=" * 72)
    print("PER-DATASET DEEP INSPECTION")
    print("=" * 72)

    for group_key in sorted(dataset_groups.keys()):
        files = dataset_groups[group_key]
        print(f"\n--- {group_key} ---")

        total_size = sum(os.path.getsize(f) for f in files if os.path.exists(f))
        print(f"  Files: {len(files)}  |  Size: {human_size(total_size)}")

        ext_counts = defaultdict(int)
        for f in files:
            ext = os.path.splitext(f)[1].lower() or "(no ext)"
            ext_counts[ext] += 1
        print(f"  Extensions: {dict(sorted(ext_counts.items(), key=lambda x: -x[1]))}")

        classification = classify_directory_set(files)
        print(f"  CLASSIFICATION: {classification}")

        # weight files: full introspection since these gate the whole pipeline
        weight_files = [f for f in files if is_weight(f)]
        for wf in weight_files:
            print(f"  WEIGHT: {wf}  ({human_size(os.path.getsize(wf))})")
            print(f"          -> {inspect_weight_file(wf)}")

        # image / mask content inspection (sample a few real files)
        image_files = [f for f in files if is_image(f) and not looks_like_mask_name(os.path.basename(f))]
        mask_files = [f for f in files if is_image(f) and looks_like_mask_name(os.path.basename(f))]

        if image_files:
            print(f"  Sample image inspection ({min(SAMPLE_N, len(image_files))} of {len(image_files)}):")
            for f in image_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        if mask_files:
            print(f"  Sample mask inspection ({min(SAMPLE_N, len(mask_files))} of {len(mask_files)}):")
            for f in mask_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    binary_hint = "binary-like (0/255 or 0/1)" if mx <= 255 and mn == 0 and (mx in (1, 255)) else "check manually"
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]  {binary_hint}")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        # image/mask count parity check
        if image_files and mask_files:
            if len(image_files) != len(mask_files):
                print(f"  !! COUNT MISMATCH: {len(image_files)} images vs {len(mask_files)} masks -- some are likely unpaired")
            else:
                print(f"  Image/mask counts match: {len(image_files)} each")

        # CSV preview
        csv_files = [f for f in files if f.lower().endswith(".csv")]
        for cf in csv_files:
            print(f"  CSV: {cf}")
            for row in preview_csv(cf):
                print(f"    {row}")

    print("\n" + "=" * 72)
    print("PATHS COMMONLY HARDCODED IN CHAKRAMODEL SCRIPTS -- verify these")
    print("=" * 72)
    candidate_paths = [
        "/kaggle/input/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
        "/kaggle/input/datasets/gokulraj324/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-yolo-combo-dataset",
        "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb",
        "/kaggle/working",
    ]
    for p in candidate_paths:
        exists = os.path.exists(p)
        print(f"  [{'OK  ' if exists else 'MISSING'}] {p}")

    print("\nDone. Use the CLASSIFICATION lines above to know which scripts can run as-is")
    print("(SEGMENTATION datasets need bbox generation before YOLO training; only a")
    print("YOLO-DETECTION dataset with images/train+val AND label .txt files is ready to train on).")

TOP-LEVEL CONTENTS OF /kaggle/input  (1 entries)
  [DIR ] datasets

PER-DATASET DEEP INSPECTION

--- datasets/balraj98/cvcclinicdb ---
  Files: 2450  |  Size: 309.5MB
  Extensions: {'.tif': 1224, '.png': 1224, '.csv': 2}
  CLASSIFICATION: PLAIN IMAGE COLLECTION (no masks, no labels)
  Sample image inspection (5 of 2448):
    274.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    315.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    123.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    130.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    508.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
  CSV: /kaggle/input/datasets/balraj98/cvcclinicdb/class_dict.csv
    ['class_names', 'r', 'g', 'b']
    ['background', '0', '0', '0']
    ['polyp', '255', '255', '255']
  CSV: /kaggle/input/datasets/balraj98/cvcclinicdb/metadata.csv
    ['frame_id', 'sequence_id', 'tif_image_path', 'tif_mas